In [0]:
import pandas as pd

In [0]:
%pip install openpyxl


In [0]:
catalog_path = "/Volumes/edtech_project/edtech_bronze/edtech_project/course_catalog/course_catalog.xlsx"


catalog_pdf = pd.read_excel(catalog_path)

#no duplicate course id's

assert catalog_pdf['course_id'].duplicated().sum() == 0, "Duplicate course_ids found!"

#is_active must be boolean
catalog_pdf['is_active'] = catalog_pdf['is_active'].astype(bool)

#avg_rating btwn 0 and 5
catalog_pdf = catalog_pdf[(catalog_pdf['avg_rating'] >= 0) & (catalog_pdf['avg_rating'] <= 5)]

#convert to spark df
catalog_df = spark.createDataFrame(catalog_pdf)



In [0]:
from pyspark.sql.functions import lit, current_timestamp
#add metadata

catalog_df = (catalog_df
    .withColumn("_source_file", lit(catalog_path))
    .withColumn("_load_ts", current_timestamp())
    .withColumn("_schema_version", lit("v1"))
    .withColumn("_last_modified_ts", current_timestamp())
)

#write course catalog bronze table
(catalog_df.write
    .format("delta")
    .mode("overwrite")   # overwrite each time XLSX is updated
    .saveAsTable("course_catalog_bronze")
)

In [0]:

%sql
-- display the table
SELECT * 
FROM course_catalog_bronze
LIMIT 5;

In [0]:
%sql
select count(*) from course_catalog_bronze;